# Soup Lab Bulk H2D

Profile-driven post-hackathon experiment on free Kaggle T4.
Pinned control and candidate use 28-layer activation offload.
The candidate groups saved host views into slabs and bulk H2D transfers, using existing GPU staging.
First run the full patched tests and a same-session solution-only screen for each side.
If the screen stays within 500 MiB and is not over 1 s slower, run each original 2+2 ABBA and all ten checks.
Screens and profiles are diagnostic; only original full-command measurements qualify.
No paid GPU, old competition notebook, harness or task modification.
Outputs: bulk-comparison/COMPARISON.json, SCREEN.json, source-bound raw archives.


In [ ]:
import os, subprocess, sys
from pathlib import Path
CONTROL = '06dc4b1957fbe1e23b4b32fa420d3205890fa553'
CANDIDATE = '77cfc4d7ef293b1d45e2dab06b5428d178f1faa1'
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
DRIVER = Path('/kaggle/working/bulk-comparison-driver')
OUT = Path('/kaggle/working/bulk-comparison')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not DRIVER.exists() and not OUT.exists(), 'Use a fresh session'
run(['git', 'init', str(DRIVER)])
run(['git', '-C', str(DRIVER), 'config', 'core.autocrlf', 'false'])
run(['git', '-C', str(DRIVER), 'config', 'core.eol', 'lf'])
run(['git', '-C', str(DRIVER), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(DRIVER), 'fetch', '--depth', '1', 'origin', CANDIDATE])
run(['git', '-C', str(DRIVER), 'checkout', '--detach', 'FETCH_HEAD'])
assert subprocess.check_output(['git', '-C', str(DRIVER), 'rev-parse', 'HEAD'], text=True).strip() == CANDIDATE
gpus = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpus)
assert 'T4' in gpus, 'Use Tesla T4'


In [ ]:
# Installation is outside the official timed child processes.
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])


In [ ]:
# No torch import/context in this notebook process. GPU work is in ended children.
run([sys.executable, 'research/run_bulk_transfer_comparison.py', '--control', CONTROL,
     '--candidate', CANDIDATE, '--out', str(OUT)], cwd=DRIVER)


In [ ]:
import json
report = json.loads((OUT / 'COMPARISON.json').read_text())
for name, result in report['results'].items():
    print(name, result['commit'], 'all_checks:', result['all_checks'])
    print(json.dumps(result.get('summary', result), indent=2))
print(report['limitations'])
